# Demografía bonaerense
## Nacimientos, envejecimiento y dependencia potencial en los 135 partidos

Proyecto de **Programación Orientada a Datos**.

### Problemática
La disminución de los nacimientos inscriptos y el aumento del peso relativo de las personas mayores pueden modificar la estructura por edades de la Provincia de Buenos Aires. Estos cambios pueden generar nuevas demandas de cuidado, salud, educación y protección social, con diferencias importantes entre los 135 partidos.

### Objetivo
Analizar descriptivamente la evolución de los nacimientos y defunciones inscriptos, el envejecimiento y la dependencia potencial en los 135 partidos bonaerenses, identificar diferencias territoriales y aportar evidencia útil para discutir desafíos de planificación pública, **sin inferir causalidad ni presentar proyecciones de población**.

### Pregunta de análisis
**¿Cómo se relacionan la caída de los nacimientos inscriptos, el envejecimiento y la dependencia potencial entre los partidos bonaerenses?**

El notebook realiza el flujo completo del proyecto: descarga fuentes oficiales, controla calidad, construye indicadores, genera conclusiones y crea un tablero HTML listo para publicar con GitHub Pages.


## 1. Configuración y fuentes


In [ ]:
# Configuración general, fuentes y funciones auxiliares
import re
import unicodedata
import json
import html
import hashlib
import shutil
import time
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import requests
import pandas as pd
from IPython.display import display, HTML, FileLink

# Fuentes oficiales
URL_POBLACION = "https://www.indec.gob.ar/ftp/cuadros/poblacion/c2022_bsas_est_c1_2.xlsx"
URL_EDADES = "https://www.indec.gob.ar/ftp/cuadros/poblacion/c2022_bsas_est_c5_2.xlsx"
URL_ENVEJECIMIENTO = "https://www.indec.gob.ar/ftp/cuadros/poblacion/c2022_bsas_est_c8_2.xlsx"
URL_DEPENDENCIA = "https://www.indec.gob.ar/ftp/cuadros/poblacion/c2022_bsas_est_c9_2.xlsx"
URL_NAC = (
    "https://catalogo.datos.gba.gob.ar/dataset/940fc93f-a2c3-431b-ac51-ca84507bc50d"
    "/resource/64f69263-b911-47f0-9da8-e6b7ee43e584/download/nacimientos_mensuales_012020-092026.csv"
)
URL_DEF = (
    "https://catalogo.datos.gba.gob.ar/dataset/194f8b8c-d936-44dc-87a0-cfedf4563422"
    "/resource/b1d0fa73-399a-45f9-934b-998730cec8da/download/defunciones_mensuales_012020-092026.csv"
)

# Parámetros del análisis
FECHA_CORTE = pd.Timestamp("2026-09-01")
FECHA_INICIO_ESCENARIO = pd.Timestamp("2022-06-01")
ANIO_BASE = 2022
ANIO_FINAL = 2025
MIN_NACIMIENTOS_RANKING = 100

PROBLEMATICA = (
    "La disminución de los nacimientos inscriptos y el aumento del peso relativo de las "
    "personas mayores pueden modificar la estructura por edades de la Provincia de Buenos Aires. "
    "Estos cambios pueden generar nuevas demandas de cuidado, salud, educación y protección social, "
    "con diferencias importantes entre los 135 partidos."
)

OBJETIVO = (
    "Analizar descriptivamente la evolución de los nacimientos y defunciones inscriptos, el "
    "envejecimiento y la dependencia potencial en los 135 partidos bonaerenses, identificar "
    "diferencias territoriales y aportar evidencia útil para discutir desafíos de planificación "
    "pública, sin inferir causalidad ni presentar proyecciones de población."
)

PREGUNTA_ANALISIS = (
    "¿Cómo se relacionan la caída de los nacimientos inscriptos, el envejecimiento y la "
    "dependencia potencial entre los partidos bonaerenses?"
)

# Salidas del proyecto
SALIDA = Path("analisis_bonaerense")
DOCS_DIR = Path("docs")
INDEX_HTML = DOCS_DIR / "index.html"
SALIDA.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)


def texto(valor):
    return re.sub(r"\s+", " ", str(valor).replace("\xa0", " ")).strip()


def clave_nombre(valor):
    # Solo para vincular las hojas de edades. Conservamos el nombre original.
    s = unicodedata.normalize("NFKD", texto(valor))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r"[^a-z0-9]", "", s)


def numero(serie):
    # '-' es cero; '///' significa que el dato no corresponde, NO cero.
    s = serie.astype("string").str.strip().replace({"-": "0", "///": pd.NA})
    return pd.to_numeric(s, errors="raise")


def descargar_bytes(url, intentos=4, timeout=90):
    """Descarga una fuente con reintentos ante fallas transitorias de conexión."""
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return r.content
        except requests.exceptions.RequestException as exc:
            ultimo_error = exc
            if intento < intentos:
                espera = 5 * intento
                print(f"⚠️ Falló la descarga ({intento}/{intentos}). Reintento en {espera}s...")
                time.sleep(espera)
    raise RuntimeError(f"No se pudo descargar la fuente después de {intentos} intentos: {url}") from ultimo_error


def descargar_excel(url):
    contenido = descargar_bytes(url)
    if not contenido.startswith(b"PK"):
        raise ValueError(f"El servidor no devolvió un Excel válido: {url}")
    return pd.ExcelFile(BytesIO(contenido), engine="openpyxl")


def comprobar_claves(tabla, columnas, cantidad=None):
    if tabla[columnas].isna().any().any():
        raise ValueError(f"Hay claves vacías en {columnas}")
    if tabla.duplicated(columnas).any():
        raise ValueError(f"Hay claves repetidas en {columnas}; revisar, no borrar a ciegas")
    if cantidad is not None and len(tabla) != cantidad:
        raise ValueError(f"Se esperaban {cantidad} filas y se obtuvieron {len(tabla)}")


display(HTML("<div style='background:#111827;color:#f1f5f9;padding:14px 18px;border-radius:10px;border:1px solid #334155;font-family:Arial'><b>Demografía bonaerense · análisis de los 135 partidos</b></div>"))


## 2. Marco teórico

Este trabajo se apoya en conceptos de demografía descriptiva para interpretar cambios en la estructura por edades.

### Transición demográfica y envejecimiento
Cuando disminuyen los nacimientos y aumenta la supervivencia a edades avanzadas, la estructura etaria puede desplazarse hacia una mayor proporción de personas mayores. Esto no ocurre de forma idéntica en todos los territorios y puede convivir con migraciones, diferencias económicas y cambios en la composición de los hogares.

### Nacimientos inscriptos
Los archivos provinciales utilizados registran **inscripciones**. Por eso, en este proyecto se habla de *nacimientos inscriptos* y no de una tasa oficial de natalidad o fecundidad por residencia. La distinción evita atribuir a los registros administrativos una interpretación que la fuente no garantiza.

### Índice de envejecimiento
El indicador publicado por INDEC expresa la cantidad de personas de **65 años o más por cada 100 menores de 15 años**. Un valor más alto indica un mayor peso relativo de las edades mayores respecto de las edades jóvenes.

### Dependencia potencial
La dependencia potencial es una relación demográfica por edad:

\[
\text{Dependencia potencial} =
\frac{\text{Población 0–14} + \text{Población 65+}}
{\text{Población 15–64}} \times 100
\]

Se descompone en:

- **Dependencia joven:** menores de 15 por cada 100 personas de 15–64.
- **Dependencia de mayores:** personas de 65+ por cada 100 personas de 15–64.

Es un indicador **potencial**: no significa que toda persona de 15–64 esté ocupada ni que toda persona de 65+ sea económicamente dependiente.

### Saldo registral
Se calcula como nacimientos inscriptos menos defunciones inscriptas. No equivale automáticamente al crecimiento natural de residentes ni al cambio total de población, porque no incorpora migraciones y la documentación de las fuentes no permite asumir residencia de manera directa.

### Enfoque del trabajo
El análisis es descriptivo. Busca identificar patrones, diferencias territoriales y relaciones entre indicadores. No prueba causalidad y no sustituye proyecciones demográficas oficiales.


## 3. Censo 2022: población y estructura por edades


In [ ]:
# Población total 2010 y 2022. Quitamos carátulas, notas y subtotales.
x_poblacion = descargar_excel(URL_POBLACION)
bruto = pd.read_excel(x_poblacion, sheet_name="Cuadro1.2", header=None)
codigos = bruto[0].astype("string").str.strip()
filas_partidos = codigos.str.fullmatch(r"06\d{3}", na=False)
censo_poblacion = bruto.loc[filas_partidos, [0, 1, 2, 3]].copy()
censo_poblacion.columns = ["municipio_id", "municipio_nombre", "poblacion_2010", "poblacion_2022"]
censo_poblacion["municipio_id"] = censo_poblacion["municipio_id"].astype("string").str.strip()
censo_poblacion["municipio_nombre"] = censo_poblacion["municipio_nombre"].map(texto)
for col in ["poblacion_2010", "poblacion_2022"]:
    censo_poblacion[col] = numero(censo_poblacion[col]).astype("Int64")
comprobar_claves(censo_poblacion, ["municipio_id"], 135)
if censo_poblacion["poblacion_2022"].isna().any() or (censo_poblacion["poblacion_2022"] <= 0).any():
    raise ValueError("Falta población censal 2022 o hay cantidades no válidas")
censo_poblacion["fecha_censo"] = "2022-05-18"
# Cambio de límites: 2010 incluye Lezama dentro de Chascomús.
censo_poblacion["comparable_2010_2022"] = (
    censo_poblacion["poblacion_2010"].notna()
    & ~censo_poblacion["municipio_id"].isin(["06218", "06466"])
)
censo_poblacion["variacion_poblacion_pct_2010_2022"] = (
    (censo_poblacion["poblacion_2022"] / censo_poblacion["poblacion_2010"] - 1) * 100
).where(censo_poblacion["comparable_2010_2022"])
total_provincia = numero(bruto.loc[bruto[1].map(texto).eq("Total"), 3]).iloc[0]
if censo_poblacion["poblacion_2022"].sum() != total_provincia:
    raise ValueError("La suma de partidos no coincide con el total provincial del cuadro 1.2")

# Edades. Cada partido tiene una hoja, pero su número NO es municipio_id.
referencia = {clave_nombre(r.municipio_nombre): (r.municipio_id, r.municipio_nombre)
              for r in censo_poblacion.itertuples()}
if len(referencia) != 135:
    raise ValueError("Hay nombres de partidos ambiguos después de normalizarlos")
x_edades = descargar_excel(URL_EDADES)
hojas = [s for s in x_edades.sheet_names if re.fullmatch(r"Cuadro5\.2\.[1-9]\d*", s)]
registros, controles_edades = [], []
for hoja in hojas:
    tabla = pd.read_excel(x_edades, sheet_name=hoja, header=None)
    hallazgo = re.search(r", partido (.*?)\. Total", texto(tabla.iat[1, 0]))
    if hallazgo is None or clave_nombre(hallazgo.group(1)) not in referencia:
        raise ValueError(f"No se pudo identificar el partido de {hoja}")
    codigo, nombre = referencia[clave_nombre(hallazgo.group(1))]
    etiquetas = tabla[0].map(texto)
    # Tomamos edades simples 0...99 y el grupo abierto 100 y más.
    # Excluimos 0-4, 5-9, etc. porque son subtotales de las mismas personas.
    edades = etiquetas.str.fullmatch(r"\d{1,2}|100 y más")
    parte = tabla.loc[edades, [0, 1, 2, 3]].copy()
    parte.columns = ["edad", "poblacion", "poblacion_viviendas_particulares", "poblacion_viviendas_colectivas"]
    parte["edad"] = parte["edad"].map(texto).replace({"100 y más": "100"}).astype(int)
    for col in parte.columns[1:]:
        parte[col] = numero(parte[col]).astype("Int64")
        if parte[col].isna().any() or (parte[col] < 0).any():
            raise ValueError(f"Cantidad de población no válida en {hoja}: {col}")
    if set(parte["edad"]) != set(range(101)):
        raise ValueError(f"Faltan edades en {hoja}")
    if not parte["poblacion"].eq(parte["poblacion_viviendas_particulares"] + parte["poblacion_viviendas_colectivas"]).all():
        raise ValueError(f"No coinciden los componentes de población en {hoja}")
    total_hoja = numero(tabla.loc[etiquetas.eq("Total"), 1]).iloc[0]
    if parte["poblacion"].sum() != total_hoja:
        raise ValueError(f"Las edades no suman el total declarado en {hoja}")
    parte["municipio_id"], parte["municipio_nombre"], parte["anio"] = codigo, nombre, 2022
    parte["edad_abierta"] = parte["edad"].eq(100)
    parte["grupo_edad"] = pd.cut(parte["edad"], [-1, 14, 64, 100], labels=["0_14", "15_64", "65_mas"]).astype("string")
    registros.append(parte)
    controles_edades.append({"municipio_id": codigo, "total_cuadro_edades": int(total_hoja)})

censo_edades = pd.concat(registros, ignore_index=True)
censo_edades["municipio_id"] = censo_edades["municipio_id"].astype("string")
censo_edades = censo_edades.sort_values(["municipio_id", "edad"]).reset_index(drop=True)
comprobar_claves(censo_edades, ["municipio_id", "edad"], 135 * 101)
if set(censo_edades["municipio_id"]) != set(censo_poblacion["municipio_id"]):
    raise ValueError("Los partidos de edades no coinciden con los de población")


## 4. Envejecimiento y dependencia potencial


In [ ]:
# Índice oficial histórico. Conservamos faltantes y definición de la fuente.
x_envejecimiento = descargar_excel(URL_ENVEJECIMIENTO)
bruto_indice = pd.read_excel(x_envejecimiento, sheet_name="Cuadro 8.2", header=None)
mascara = bruto_indice[0].astype("string").str.strip().str.fullmatch(r"06\d{3}", na=False)
indices = bruto_indice.loc[mascara, list(range(8))].copy()
indices.columns = ["municipio_id", "nombre_fuente", 1970, 1980, 1991, 2001, 2010, 2022]
indices["municipio_id"] = indices["municipio_id"].astype("string").str.strip()
comprobar_claves(indices, ["municipio_id"], 135)
if set(indices["municipio_id"]) != set(censo_poblacion["municipio_id"]):
    raise ValueError("Los partidos del índice no coinciden con el censo")
censo_envejecimiento = indices.drop(columns="nombre_fuente").melt(
    id_vars="municipio_id", var_name="anio", value_name="indice_envejecimiento_oficial")
censo_envejecimiento["anio"] = censo_envejecimiento["anio"].astype(int)
censo_envejecimiento["indice_envejecimiento_oficial"] = numero(censo_envejecimiento["indice_envejecimiento_oficial"]).astype("Float64")
censo_envejecimiento = censo_envejecimiento.merge(
    censo_poblacion[["municipio_id", "municipio_nombre"]], on="municipio_id", validate="many_to_one")
censo_envejecimiento["universo_2022"] = "Viviendas particulares; consultar notas de la fuente para otros censos"
censo_envejecimiento = censo_envejecimiento.sort_values(["municipio_id", "anio"]).reset_index(drop=True)
comprobar_claves(censo_envejecimiento, ["municipio_id", "anio"], 135 * 6)

# Base con una fila por partido para cruzar después.
grupos = censo_edades.groupby(["municipio_id", "grupo_edad"], observed=True)["poblacion"].sum().unstack()
grupos = grupos.rename(columns={"0_14": "poblacion_0_14_2022", "15_64": "poblacion_15_64_2022", "65_mas": "poblacion_65_mas_2022"}).reset_index()
censo_base = censo_poblacion.merge(grupos, on="municipio_id", validate="one_to_one")
censo_base = censo_base.merge(pd.DataFrame(controles_edades), on="municipio_id", validate="one_to_one")
censo_base["diferencia_total_y_cuadro_edades"] = censo_base["poblacion_2022"] - censo_base["total_cuadro_edades"]
for grupo in ["0_14", "15_64", "65_mas"]:
    # Denominador: población del cuadro de edades, no el total de otro universo.
    censo_base[f"pct_{grupo}_sobre_cuadro_edades_2022"] = (
        100 * censo_base[f"poblacion_{grupo}_2022"] / censo_base["total_cuadro_edades"])
indice_2022 = censo_envejecimiento.loc[censo_envejecimiento["anio"].eq(2022),
    ["municipio_id", "indice_envejecimiento_oficial"]].rename(
    columns={"indice_envejecimiento_oficial": "indice_envejecimiento_oficial_2022"})
censo_base = censo_base.merge(indice_2022, on="municipio_id", validate="one_to_one")
censo_base = censo_base.sort_values("municipio_id").reset_index(drop=True)
comprobar_claves(censo_base, ["municipio_id"], 135)


# Dependencia potencial oficial por partido: personas 0-14 y 65+ por cada 100 de 15-64.
x_dependencia = descargar_excel(URL_DEPENDENCIA)
bruto_dependencia = pd.read_excel(x_dependencia, sheet_name='Cuadro 9.2.bis', header=None)
mask_dep = bruto_dependencia[0].astype('string').str.strip().str.fullmatch(r'06\d{3}', na=False)
dep = bruto_dependencia.loc[mask_dep, [0, 1, 6, 7]].copy()
dep.columns = ['municipio_id', 'nombre_fuente', 2010, 2022]
dep['municipio_id'] = dep['municipio_id'].astype('string').str.strip()
comprobar_claves(dep, ['municipio_id'], 135)
if set(dep['municipio_id']) != set(censo_base['municipio_id']): raise ValueError('Dependencia potencial no coincide con los 135 partidos')
for a in [2010, 2022]: dep[a] = numero(dep[a]).astype('Float64')
dep = dep.merge(censo_base[['municipio_id','municipio_nombre']], on='municipio_id', validate='one_to_one')
dep['cambio_2010_2022'] = dep[2022] - dep[2010]
dep_prov = bruto_dependencia.loc[bruto_dependencia[1].map(texto).eq('Total')].iloc[0]
dep_prov_2010, dep_prov_2022 = float(dep_prov[6]), float(dep_prov[7])
censo_base['dependencia_joven_2022'] = 100*censo_base['poblacion_0_14_2022']/censo_base['poblacion_15_64_2022']
censo_base['dependencia_mayores_2022'] = 100*censo_base['poblacion_65_mas_2022']/censo_base['poblacion_15_64_2022']
censo_base['dependencia_total_calculada_2022'] = censo_base['dependencia_joven_2022']+censo_base['dependencia_mayores_2022']
censo_base = censo_base.merge(dep[['municipio_id', 2010, 2022, 'cambio_2010_2022']], on='municipio_id', validate='one_to_one')
censo_base = censo_base.rename(columns={2010:'dependencia_potencial_oficial_2010',2022:'dependencia_potencial_oficial_2022','cambio_2010_2022':'cambio_dependencia_potencial_2010_2022'})


## 5. Nacimientos, defunciones y controles de calidad


In [ ]:
# Nacimientos y defunciones: descarga, limpieza y análisis temporal.
calidad_registros, huellas = [], []

def limpiar_eventos(url, tipo):
    contenido = descargar_bytes(url)
    huellas.append({'fuente': tipo, 'url': url,
                    'sha256': hashlib.sha256(contenido).hexdigest()})
    original = pd.read_csv(BytesIO(contenido), sep=';', encoding='utf-8-sig',
                           dtype={'municipio_id': 'string'})
    d = original.rename(columns={'municipio': 'municipio_nombre',
        'defunciones_femeninas': 'femenino', 'defunciones_masculinas': 'masculino'}).copy()
    requeridas = ['municipio_id','municipio_nombre','anio','mes','femenino','masculino','total']
    if not set(requeridas).issubset(d):
        raise ValueError(f'Cambiaron las columnas de {tipo}: {list(d.columns)}')
    d = d[requeridas].copy()
    d['municipio_id'] = d['municipio_id'].str.strip().str.replace(r'\.0$', '', regex=True).str.zfill(5)
    d['municipio_nombre'] = d['municipio_nombre'].map(texto)
    for col in ['anio','mes','femenino','masculino','total']:
        d[col] = pd.to_numeric(d[col], errors='raise')
        if d[col].isna().any() or (d[col] < 0).any() or (d[col] % 1 != 0).any():
            raise ValueError(f'Valores nulos, negativos o no enteros en {tipo}/{col}')
        d[col] = d[col].astype('int64')
    if not d['mes'].between(1, 12).all():
        raise ValueError(f'Meses no válidos en {tipo}')
    if not d['municipio_id'].str.fullmatch(r'06\d{3}').all():
        raise ValueError(f'Códigos municipales no válidos en {tipo}')
    if not set(d['municipio_id']).issubset(set(censo_base['municipio_id'])):
        raise ValueError(f'Hay partidos de {tipo} sin correspondencia en el censo')
    d['fecha'] = pd.to_datetime(dict(year=d['anio'], month=d['mes'], day=1))
    exactos = int(d.duplicated().sum())
    d = d.drop_duplicates()  # solo duplicados idénticos, nunca claves en conflicto
    comprobar_claves(d, ['municipio_id','fecha'])
    fuera = d.loc[(d['fecha'] > FECHA_CORTE) | (d['fecha'] < '2020-01-01')].copy()
    fuera.to_csv(SALIDA / f'{tipo}_fuera_periodo.csv', index=False, encoding='utf-8-sig')
    d = d.loc[d['fecha'].between('2020-01-01', FECHA_CORTE)].copy()
    # La clave une las fuentes; el nombre oficial se toma del censo.
    d = d.drop(columns='municipio_nombre').merge(
        censo_base[['municipio_id','municipio_nombre']], on='municipio_id', validate='many_to_one')
    d['alerta_componentes'] = d['total'].ne(d['femenino'] + d['masculino'])
    med = d.groupby(['municipio_id','anio'])['total'].transform('median')
    # Regla descriptiva, no prueba de un error: los valores se conservan.
    d['alerta_caida'] = (med >= 30) & (d['total'] < 0.5 * med)
    d['total_cero'] = d['total'].eq(0)
    d = d.sort_values(['municipio_id','fecha']).reset_index(drop=True)
    calendario = pd.MultiIndex.from_product(
        [censo_base['municipio_id'], pd.date_range('2020-01-01', FECHA_CORTE, freq='MS')],
        names=['municipio_id','fecha'])
    faltantes = calendario.difference(pd.MultiIndex.from_frame(d[['municipio_id','fecha']]))
    pd.DataFrame(faltantes.tolist(), columns=['municipio_id','fecha']).to_csv(
        SALIDA / f'{tipo}_meses_faltantes.csv', index=False)
    calidad_registros.append({'fuente':tipo, 'filas_originales':len(original),
        'duplicados_exactos_eliminados':exactos, 'filas_fuera_periodo':len(fuera),
        'filas_limpias':len(d), 'meses_faltantes':len(faltantes),
        'incoherencias_componentes':int(d['alerta_componentes'].sum()),
        'alertas_caidas':int(d['alerta_caida'].sum()), 'registros_en_cero':int(d['total_cero'].sum())})
    return d

nacimientos = limpiar_eventos(URL_NAC, 'nacimientos')
defunciones = limpiar_eventos(URL_DEF, 'defunciones')
calidad = pd.DataFrame(calidad_registros)

# Unión por partido y mes. Reindexar deja faltantes como NaN, nunca como cero.
calendario = pd.MultiIndex.from_product(
    [censo_base['municipio_id'], pd.date_range('2020-01-01', FECHA_CORTE, freq='MS')],
    names=['municipio_id','fecha'])
mensual = pd.DataFrame(index=calendario).reset_index()
for fuente, etiqueta in [(nacimientos,'nacimientos'),(defunciones,'defunciones')]:
    t = fuente[['municipio_id','fecha','total','alerta_caida','alerta_componentes']].rename(
        columns={'total':etiqueta, 'alerta_caida':f'alerta_{etiqueta}',
                 'alerta_componentes':f'componentes_{etiqueta}'})
    mensual = mensual.merge(t, on=['municipio_id','fecha'], how='left', validate='one_to_one')
mensual = mensual.merge(censo_base[['municipio_id','municipio_nombre','poblacion_2022']],
                         on='municipio_id', how='left', validate='many_to_one')
mensual['anio'], mensual['mes'] = mensual['fecha'].dt.year, mensual['fecha'].dt.month
mensual['saldo_registral'] = mensual['nacimientos'] - mensual['defunciones']
mensual['mes_con_ambas_fuentes'] = mensual[['nacimientos','defunciones']].notna().all(axis=1)

# No se llama tasa de natalidad: el denominador queda fijo en 2022 y el numerador
# es inscripción municipal, sin residencia confirmada por la documentación.
anual = mensual.groupby(['municipio_id','anio'], as_index=False).agg(
    nacimientos=('nacimientos',lambda s:s.sum(min_count=1)),
    defunciones=('defunciones',lambda s:s.sum(min_count=1)),
    meses_nacimientos=('nacimientos','count'), meses_defunciones=('defunciones','count'),
    alertas_nacimientos=('alerta_nacimientos','sum'), alertas_defunciones=('alerta_defunciones','sum'))
anual['anio_completo'] = anual['meses_nacimientos'].eq(12) & anual['meses_defunciones'].eq(12)
anual['saldo_registral'] = anual['nacimientos'] - anual['defunciones']
anual = anual.merge(censo_base, on='municipio_id', validate='many_to_one')
anual['inscripciones_nac_por_1000_hab_censo2022'] = 1000 * anual['nacimientos'] / anual['poblacion_2022']
completos = anual[anual['anio_completo'] & anual['anio'].between(2020, ANIO_FINAL)].copy()
provincial = completos.groupby('anio', as_index=False).agg(
    nacimientos=('nacimientos','sum'), defunciones=('defunciones','sum'),
    partidos=('municipio_id','nunique'))
# Nunca presentar como total de provincia una suma con partidos faltantes.
if not provincial['partidos'].eq(135).all() or set(provincial['anio']) != set(range(2020, 2026)):
    raise ValueError('Falta cobertura para comparar años completos en los 135 partidos')
provincial['saldo_registral'] = provincial['nacimientos'] - provincial['defunciones']
provincial['inscripciones_por_1000_hab_censo2022'] = 1000 * provincial['nacimientos'] / censo_base['poblacion_2022'].sum()

base = completos[completos['anio'].eq(ANIO_BASE)][['municipio_id','nacimientos']].rename(columns={'nacimientos':'nacimientos_base'})
fin = completos[completos['anio'].eq(ANIO_FINAL)][['municipio_id','nacimientos','defunciones','saldo_registral','alertas_nacimientos']]
ranking = fin.merge(base,on='municipio_id',validate='one_to_one').merge(censo_base,on='municipio_id',validate='one_to_one')
ranking['variacion_nacimientos_pct_2022_2025'] = (100 * (ranking['nacimientos']/ranking['nacimientos_base']-1)).where(ranking['nacimientos_base']>0)
ranking['apto_ranking'] = ranking['nacimientos_base'].ge(MIN_NACIMIENTOS_RANKING)
ranking = ranking.sort_values('variacion_nacimientos_pct_2022_2025').reset_index(drop=True)

# Mismo lapso para el año parcial, con control de cobertura.
ult_anio, ult_mes = FECHA_CORTE.year, FECHA_CORTE.month
parcial = mensual[mensual['anio'].isin([ult_anio-1,ult_anio]) & mensual['mes'].le(ult_mes)]
if not parcial.groupby('anio')['mes_con_ambas_fuentes'].sum().eq(135*ult_mes).all():
    raise ValueError('Falta cobertura para comparar los mismos meses de 2025 y 2026')
ytd = parcial.groupby('anio',as_index=False)[['nacimientos','defunciones']].sum(min_count=1)
ytd['saldo_registral'] = ytd['nacimientos']-ytd['defunciones']

# Índices históricos: publicados por INDEC, sin fabricar promedios provinciales.
ind = censo_envejecimiento.pivot(index='municipio_id',columns='anio',values='indice_envejecimiento_oficial')
ind['cambio_2010_2022'] = ind[2022] - ind[2010]
ind = ind.merge(censo_base[['municipio_id','municipio_nombre']],on='municipio_id',validate='one_to_one')
comparables_indice = ind.dropna(subset=[2010,2022])
# Total provincial oficial, no la media simple de los municipios.
fila_prov = bruto_indice.loc[bruto_indice[1].map(texto).eq('Total')].iloc[0]
indice_prov_2010, indice_prov_2022 = float(fila_prov[6]), float(fila_prov[7])

# Escenario aritmético sin migraciones, separado de los resultados observados.
# Supone que las inscripciones equivalen a eventos de residentes: no está verificado.
# El punto inicial es 18/05/2022 y el primer mes que se agrega es junio de 2022.
# No incorporamos el 19-31 de mayo porque no tenemos fechas diarias.
escenario = mensual[mensual['fecha'].ge(FECHA_INICIO_ESCENARIO)].sort_values(['municipio_id','fecha']).copy()
escenario['periodo_completo_hasta_fecha'] = escenario.groupby('municipio_id')['mes_con_ambas_fuentes'].cummin()
escenario['saldo_acumulado_desde_junio2022'] = escenario.groupby('municipio_id')['saldo_registral'].cumsum()
escenario['poblacion_escenario_sin_migraciones'] = (
    escenario['poblacion_2022']+escenario['saldo_acumulado_desde_junio2022']
).where(escenario['periodo_completo_hasta_fecha'])
escenario['supuestos'] = 'Sin migraciones; inscripción como proxy de residencia; omite 19-31 mayo 2022'


## 6. Análisis de dependencia potencial


In [ ]:
# Análisis específico de dependencia potencial

prov_edades = censo_base[
    ["poblacion_0_14_2022", "poblacion_15_64_2022", "poblacion_65_mas_2022"]
].sum()

dep_joven_prov = (
    100 * prov_edades["poblacion_0_14_2022"] / prov_edades["poblacion_15_64_2022"]
)
dep_mayor_prov = (
    100 * prov_edades["poblacion_65_mas_2022"] / prov_edades["poblacion_15_64_2022"]
)

partidos_dep_sube = int((dep["cambio_2010_2022"] > 0).sum())
partidos_dep_baja = int((dep["cambio_2010_2022"] < 0).sum())
partidos_dep_igual = int((dep["cambio_2010_2022"] == 0).sum())

dependencia_mayores_top = (
    censo_base[
        [
            "municipio_id",
            "municipio_nombre",
            "dependencia_joven_2022",
            "dependencia_mayores_2022",
            "dependencia_potencial_oficial_2022",
        ]
    ]
    .sort_values("dependencia_mayores_2022", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

resumen_dependencia = pd.DataFrame(
    {
        "Indicador": [
            "Dependencia potencial oficial 2010",
            "Dependencia potencial oficial 2022",
            "Dependencia joven calculada 2022",
            "Dependencia de mayores calculada 2022",
            "Partidos donde aumenta la dependencia total 2010–2022",
        ],
        "Valor": [
            dep_prov_2010,
            dep_prov_2022,
            dep_joven_prov,
            dep_mayor_prov,
            partidos_dep_sube,
        ],
    }
)

display(HTML("<h3>Análisis de dependencia potencial</h3>"))
display(resumen_dependencia)
display(HTML(
    "<p><b>Lectura:</b> el indicador total provincial puede bajar aunque cambie la composición "
    "interna de las edades dependientes. Por eso se separan el componente joven y el de mayores. "
    "La dependencia potencial es una relación por edad y no una medición de empleo o inactividad.</p>"
))
display(dependencia_mayores_top)


## 7. Resultados y conclusiones descriptivas


In [ ]:
# Conclusiones generadas con los resultados, no con la hipótesis.
def num(v,dec=0):
    if pd.isna(v): return "Sin dato"
    return f"{float(v):,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct(v):
    return num(v,1) + " %"

pa = provincial.set_index("anio")
cambio_20_25 = 100 * (pa.loc[2025,"nacimientos"] / pa.loc[2020,"nacimientos"] - 1)
cambio_22_25 = 100 * (pa.loc[2025,"nacimientos"] / pa.loc[2022,"nacimientos"] - 1)
cambio_ytd = 100 * (
    ytd.set_index("anio").loc[2026,"nacimientos"] /
    ytd.set_index("anio").loc[2025,"nacimientos"] - 1
)

partidos_caida = int((ranking["variacion_nacimientos_pct_2022_2025"] < 0).sum())
partidos_saldo_neg = int((ranking["saldo_registral"] < 0).sum())
partidos_indice_sube = int((comparables_indice[2022] > comparables_indice[2010]).sum())

conclusiones = [
    f"Los nacimientos inscriptos pasan de {num(pa.loc[2020,'nacimientos'])} en 2020 a "
    f"{num(pa.loc[2025,'nacimientos'])} en 2025: una variación de {pct(cambio_20_25)}. "
    f"Desde 2022 la variación es {pct(cambio_22_25)}.",
    f"Entre 2022 y 2025 caen los nacimientos inscriptos en {partidos_caida} de 135 partidos; "
    "el comportamiento territorial no se resume en un único promedio.",
    f"En 2025 el saldo de inscripciones provincial es {num(pa.loc[2025,'saldo_registral'])} "
    f"y es negativo en {partidos_saldo_neg} partidos. Esto no equivale a medir pérdida total "
    "de población ni reemplaza estadísticas por residencia.",
    f"El índice provincial publicado de envejecimiento pasa de {num(indice_prov_2010)} a "
    f"{num(indice_prov_2022)} personas de 65+ por cada 100 menores de 15 entre 2010 y 2022. "
    f"Sube en {partidos_indice_sube} de {len(comparables_indice)} partidos con ambos valores.",
    f"La dependencia potencial oficial provincial pasa de {num(dep_prov_2010)} a "
    f"{num(dep_prov_2022)} personas en edades dependientes por cada 100 de 15–64 entre 2010 y 2022. "
    f"En {partidos_dep_sube} partidos aumentó el indicador total.",
    f"En 2022 se estiman, a partir del cuadro de edades, {num(dep_joven_prov,1)} menores de 15 "
    f"y {num(dep_mayor_prov,1)} personas de 65+ por cada 100 personas de 15–64. "
    "Separar ambos componentes evita ocultar cambios distintos dentro de la dependencia total.",
    "Si la caída de nacimientos persistiera, las cohortes futuras serían más pequeñas y, junto con "
    "el envejecimiento, podrían aumentar las presiones sobre cuidados y salud. No es una predicción: "
    "también influyen migraciones, empleo, productividad, edad de retiro y políticas públicas.",
    "Los registros utilizados no permiten afirmar una caída de la tasa anual de natalidad por "
    "residencia ni medir cuántos jóvenes hay en 2026.",
]

display(HTML("<h3>Conclusiones principales</h3>"))
for c in conclusiones:
    display(HTML(f"<p>• {html.escape(c)}</p>"))


## 8. Exportación de resultados


In [ ]:
# Exportaciones para seguir trabajando.
salidas={'censo_base_partidos_2022':censo_base,'censo_edades_partidos_2022':censo_edades,
    'censo_envejecimiento_historico':censo_envejecimiento,'censo_dependencia_potencial_historico':dep,'nacimientos_limpios':nacimientos,
    'defunciones_limpias':defunciones,'union_mensual':mensual,'resumen_anual_partidos':anual,
    'resumen_anual_provincial':provincial,'comparacion_parcial_2025_2026':ytd,
    'ranking_partidos_2022_2025':ranking,'escenario_sin_migraciones':escenario,
    'controles_limpieza':calidad}
for nombre,t in salidas.items():t.to_csv(SALIDA/f'{nombre}.csv',index=False,encoding='utf-8-sig')
pd.concat([nacimientos.assign(fuente='nacimientos'),defunciones.assign(fuente='defunciones')]).loc[
    lambda t:t['alerta_caida']|t['alerta_componentes']].to_csv(SALIDA/'registros_con_alerta.csv',index=False,encoding='utf-8-sig')
(SALIDA/'fuentes_y_huellas.json').write_text(json.dumps(huellas,ensure_ascii=False,indent=2),encoding='utf-8')
(SALIDA/'conclusiones.txt').write_text('\n\n'.join(conclusiones),encoding='utf-8')
with ZipFile('analisis_bonaerense_resultados.zip','w') as z:
    for f in sorted(SALIDA.iterdir()):z.write(f,arcname=f.name)


## 9. Dashboard web

El tablero se genera a partir de las variables calculadas en este mismo notebook. Incluye problemática, objetivo, marco teórico, análisis de dependencia, exploración territorial, conclusiones y referencias.


In [ ]:
# Preparación de datos para el HTML interactivo

INDICE_PROV_2010 = indice_prov_2010
INDICE_PROV_2022 = indice_prov_2022
DEPENDENCIA_PROV_2010 = dep_prov_2010
DEPENDENCIA_PROV_2022 = dep_prov_2022

def fmt_int(valor):
    return f"{int(round(float(valor))):,}".replace(",", ".")

def fmt_num(valor):
    valor = float(valor)
    return f"{valor:g}".replace(".", ",")

def fmt_pct(valor):
    return f"{float(valor):.1f} %".replace(".", ",")

pa = provincial.set_index("anio")
nac_2020 = pa.loc[2020, "nacimientos"]
nac_2025 = pa.loc[2025, "nacimientos"]
cambio_20_25 = 100 * (nac_2025 / nac_2020 - 1)
partidos_caida = int((ranking["variacion_nacimientos_pct_2022_2025"] < 0).sum())
porcentaje_partidos_caida = 100 * partidos_caida / 135

filas_json = []
for _, r in censo_base.sort_values("municipio_nombre").iterrows():
    cod = r["municipio_id"]
    rk = ranking.loc[ranking["municipio_id"].eq(cod)].iloc[0]
    series = anual.loc[anual["municipio_id"].eq(cod)].sort_values("anio")
    esc = escenario.loc[escenario["municipio_id"].eq(cod)].iloc[-1]
    h = ind.loc[ind["municipio_id"].eq(cod)].iloc[0]

    filas_json.append({
        "id": cod,
        "nombre": r["municipio_nombre"],
        "poblacion": int(r["poblacion_2022"]),
        "menores": int(r["poblacion_0_14_2022"]),
        "mayores": int(r["poblacion_65_mas_2022"]),
        "porcentaje_mayores": float(r["pct_65_mas_sobre_cuadro_edades_2022"]),
        "dep_joven": float(r["dependencia_joven_2022"]),
        "dep_mayores": float(r["dependencia_mayores_2022"]),
        "dep_total": float(r["dependencia_potencial_oficial_2022"]),
        "dep_2010": None if pd.isna(r["dependencia_potencial_oficial_2010"]) else float(r["dependencia_potencial_oficial_2010"]),
        "indice2010": None if pd.isna(h[2010]) else float(h[2010]),
        "indice2022": float(h[2022]),
        "cambio": None if pd.isna(rk["variacion_nacimientos_pct_2022_2025"]) else float(rk["variacion_nacimientos_pct_2022_2025"]),
        "escenario": None if pd.isna(esc["poblacion_escenario_sin_migraciones"]) else int(esc["poblacion_escenario_sin_migraciones"]),
        "series_historicas": [
            {
                "anio": int(v["anio"]),
                "nacimientos": int(v["nacimientos"]),
                "defunciones": int(v["defunciones"]),
                "saldo": int(v["saldo_registral"]),
                "alertas": int(v["alertas_nacimientos"] + v["alertas_defunciones"]),
                "completo": bool(v["anio_completo"]),
            }
            for _, v in series.iterrows()
        ],
    })

json_distritos_web = json.dumps(filas_json, ensure_ascii=False, allow_nan=False).replace("</", r"<\/")

# El scatter usa el índice oficial de envejecimiento 2022 en el eje X.
relacion_filtrada = ranking.loc[
    ranking["apto_ranking"]
    & ranking["variacion_nacimientos_pct_2022_2025"].notna()
    & ranking["indice_envejecimiento_oficial_2022"].notna()
].copy()

max_pob = relacion_filtrada["poblacion_2022"].max()
relacion_filtrada["tamano_burbuja"] = (
    10 + 24 * (relacion_filtrada["poblacion_2022"] / max_pob) ** 0.5
)

relacion_json = relacion_filtrada[[
    "municipio_nombre",
    "indice_envejecimiento_oficial_2022",
    "dependencia_mayores_2022",
    "variacion_nacimientos_pct_2022_2025",
    "poblacion_2022",
    "tamano_burbuja",
]].to_json(orient="records", force_ascii=False)

provincial_json = provincial[[
    "anio",
    "nacimientos",
    "defunciones",
    "partidos",
    "saldo_registral",
    "inscripciones_por_1000_hab_censo2022",
]].to_json(orient="records", force_ascii=False)

dependencia_prov_json = json.dumps({
    "total_2010": float(DEPENDENCIA_PROV_2010),
    "total_2022": float(DEPENDENCIA_PROV_2022),
    "joven_2022": float(dep_joven_prov),
    "mayores_2022": float(dep_mayor_prov),
    "partidos_sube": partidos_dep_sube,
    "partidos_baja": partidos_dep_baja,
}, ensure_ascii=False)

top_dependencia_json = dependencia_mayores_top[[
    "municipio_nombre",
    "dependencia_mayores_2022",
    "dependencia_joven_2022",
    "dependencia_potencial_oficial_2022",
]].sort_values("dependencia_mayores_2022").to_json(orient="records", force_ascii=False)

conclusiones_html = "".join(f"<li>{html.escape(c)}</li>" for c in conclusiones)

print("✔ Datos preparados para el dashboard.")


In [ ]:
# Construcción del sitio estático para GitHub Pages

html_template = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Demografía bonaerense · Reporte interactivo</title>
<script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
<style>
:root{
  --bg:#0b1220;--panel:#151f30;--panel2:#1c293b;--text:#f1f5f9;--muted:#c2cede;
  --green:#2dd4bf;--blue:#60a5fa;--orange:#fb923c;--line:#334155;--violet:#a78bfa
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;background:var(--bg);color:var(--text);font-family:Arial,sans-serif;line-height:1.6}
main{max-width:1180px;margin:auto;padding:24px}
header{background:linear-gradient(135deg,#111827,#17283b);border:1px solid var(--line);border-left:6px solid var(--green);border-radius:18px;padding:34px 40px;margin-bottom:18px}
h1{font-size:40px;line-height:1.15;margin:8px 0 10px}
h2{font-size:24px;margin:0 0 14px;border-bottom:1px solid var(--line);padding-bottom:10px}
h3{margin:18px 0 8px}
p,li{color:var(--muted)}
.eyebrow{font-size:12px;letter-spacing:2px;text-transform:uppercase;color:var(--green);font-weight:700}
nav{display:flex;flex-wrap:wrap;gap:8px;margin:0 0 18px}
nav a{color:var(--text);background:#111b2d;border:1px solid var(--line);padding:8px 12px;border-radius:999px;text-decoration:none;font-size:13px}
section{background:var(--panel);border:1px solid var(--line);border-radius:16px;padding:26px;margin-bottom:20px}
.problem-grid,.metric-grid,.theory-grid,.dep-grid{display:grid;gap:14px}
.problem-grid{grid-template-columns:1fr 1fr}
.metric-grid{grid-template-columns:repeat(4,1fr)}
.theory-grid{grid-template-columns:repeat(2,1fr)}
.dep-grid{grid-template-columns:repeat(4,1fr)}
.card{background:var(--panel2);border:1px solid var(--line);border-radius:12px;padding:18px}
.card strong{display:block;color:var(--blue);font-size:28px;line-height:1.2;margin:6px 0}
.card .small{font-size:13px;color:var(--muted)}
.theory-card h3{color:var(--green)}
.note{font-size:13px;color:#9fb0c8;background:#111b2d;border-left:3px solid var(--blue);padding:10px 12px;border-radius:8px}
.chart{width:100%;height:460px}
.chart.small{height:360px}
.two-col{display:grid;grid-template-columns:1fr 1fr;gap:18px}
select{width:100%;max-width:430px;background:#0f172a;color:var(--text);border:1px solid var(--line);border-radius:9px;padding:11px 14px;font-size:16px}
#partido-info-panel{background:var(--panel2);padding:20px;border-radius:12px;border:1px solid var(--line)}
.conclusions li,.references li{margin:10px 0}
.references a{color:var(--blue)}
footer{color:#94a3b8;font-size:12px;padding:8px 4px 28px}
@media(max-width:850px){
  .metric-grid,.dep-grid{grid-template-columns:repeat(2,1fr)}
  .problem-grid,.theory-grid,.two-col{grid-template-columns:1fr}
}
@media(max-width:520px){
  main{padding:12px}
  header{padding:24px}
  h1{font-size:31px}
  .metric-grid,.dep-grid{grid-template-columns:1fr}
}
</style>
</head>
<body>
<main>
<header>
  <div class="eyebrow">Programación Orientada a Datos · Provincia de Buenos Aires · 135 partidos</div>
  <h1>¿Quién sostiene una población que envejece?</h1>
  <p style="font-size:20px;color:var(--green)">Nacimientos inscriptos, envejecimiento y dependencia potencial</p>
</header>

<nav>
  <a href="#planteo">Problemática y objetivo</a>
  <a href="#marco">Marco teórico</a>
  <a href="#indicadores">Indicadores</a>
  <a href="#dependencia">Dependencia</a>
  <a href="#territorio">Territorio</a>
  <a href="#explorador">Explorador</a>
  <a href="#conclusiones">Conclusiones</a>
  <a href="#referencias">Referencias</a>
</nav>

<section id="planteo">
  <div class="eyebrow">01 · Planteo</div>
  <h2>Problemática y objetivo</h2>
  <div class="problem-grid">
    <div class="card">
      <h3>Problemática</h3>
      <p>__PROBLEMATICA__</p>
    </div>
    <div class="card">
      <h3>Objetivo</h3>
      <p>__OBJETIVO__</p>
    </div>
  </div>
  <p><b>Pregunta de análisis:</b> __PREGUNTA__</p>
</section>

<section id="marco">
  <div class="eyebrow">02 · Marco teórico</div>
  <h2>Cómo leer los indicadores demográficos</h2>
  <div class="theory-grid">
    <div class="card theory-card">
      <h3>Transición demográfica</h3>
      <p>La combinación de menos nacimientos y mayor supervivencia a edades avanzadas puede desplazar la estructura etaria hacia un mayor peso relativo de las personas mayores.</p>
    </div>
    <div class="card theory-card">
      <h3>Índice de envejecimiento</h3>
      <p>Personas de 65 años o más por cada 100 menores de 15. Permite comparar el peso relativo de ambos extremos de la estructura por edades.</p>
    </div>
    <div class="card theory-card">
      <h3>Dependencia potencial</h3>
      <p>Personas de 0–14 y 65+ por cada 100 de 15–64. Se separa en componente joven y componente de mayores para evitar que el total oculte cambios internos.</p>
    </div>
    <div class="card theory-card">
      <h3>Registros administrativos</h3>
      <p>Los archivos provinciales cuentan inscripciones. Por eso el análisis habla de nacimientos y defunciones inscriptos y no de tasas oficiales por residencia.</p>
    </div>
  </div>
  <p class="note">La dependencia potencial es una relación por edad: no equivale a cantidad real de trabajadores, jubilados, personas ocupadas ni necesidades efectivas de cuidado.</p>
</section>

<section id="indicadores">
  <div class="eyebrow">03 · Indicadores principales</div>
  <h2>Qué muestran los datos provinciales</h2>
  <div class="metric-grid">
    <div class="card"><span>Nacimientos inscriptos 2020–2025</span><strong>__CAMBIO_NAC__</strong><span class="small">De __NAC_2020__ a __NAC_2025__</span></div>
    <div class="card"><span>Partidos con caída 2022–2025</span><strong>__PARTIDOS_CAIDA__ / 135</strong><span class="small">Comparación de años completos</span></div>
    <div class="card"><span>Índice de envejecimiento</span><strong>__IND_2010__ → __IND_2022__</strong><span class="small">65+ por cada 100 menores de 15</span></div>
    <div class="card"><span>Dependencia potencial</span><strong>__DEP_2010__ → __DEP_2022__</strong><span class="small">0–14 y 65+ por cada 100 de 15–64</span></div>
  </div>
  <div id="chart-provincial" class="chart"></div>
</section>

<section id="dependencia">
  <div class="eyebrow">04 · Dependencia potencial</div>
  <h2>El total no cuenta toda la historia</h2>
  <p>El indicador oficial provincial cambia de <b>__DEP_2010__</b> en 2010 a <b>__DEP_2022__</b> en 2022. Para interpretar 2022 conviene separar sus componentes.</p>
  <div class="dep-grid">
    <div class="card"><span>Dependencia joven 2022</span><strong>__DEP_JOVEN__</strong><span class="small">0–14 por cada 100 de 15–64</span></div>
    <div class="card"><span>Dependencia de mayores 2022</span><strong>__DEP_MAYOR__</strong><span class="small">65+ por cada 100 de 15–64</span></div>
    <div class="card"><span>Partidos donde sube el total</span><strong>__DEP_SUBE__</strong><span class="small">Entre 2010 y 2022</span></div>
    <div class="card"><span>Partidos donde baja el total</span><strong>__DEP_BAJA__</strong><span class="small">Entre 2010 y 2022</span></div>
  </div>
  <div class="two-col">
    <div id="chart-dependencia-prov" class="chart small"></div>
    <div id="chart-dependencia-top" class="chart small"></div>
  </div>
  <p class="note">El gráfico provincial de componentes corresponde a 2022. El total 2010–2022 proviene del indicador oficial publicado por INDEC; no se reconstruyen componentes de 2010 que no están calculados en este proyecto.</p>
</section>

<section id="territorio">
  <div class="eyebrow">05 · Diferencias territoriales</div>
  <h2>Variación de nacimientos inscriptos vs. envejecimiento por partido</h2>
  <p>Cada burbuja representa un partido. El eje horizontal usa el <b>índice oficial de envejecimiento 2022</b>; el vertical, la variación de nacimientos inscriptos entre 2022 y 2025. El tamaño representa población censada.</p>
  <div id="chart-scatter" class="chart"></div>
  <p class="note">La relación visual es descriptiva. Una asociación entre ambos indicadores no demuestra que el envejecimiento cause la caída de nacimientos ni viceversa.</p>
</section>

<section id="explorador">
  <div class="eyebrow">06 · Explorador</div>
  <h2>Consultá cada partido</h2>
  <p>Seleccioná un partido para ver sus indicadores y su evolución registral.</p>
  <select id="partido-selector"></select>
  <div class="two-col" style="margin-top:18px">
    <div id="partido-info-panel"></div>
    <div id="chart-dependencia-partido" class="chart small"></div>
  </div>
  <div id="chart-nacimientos-partido" class="chart"></div>
</section>

<section id="conclusiones">
  <div class="eyebrow">07 · Conclusiones</div>
  <h2>Qué podemos sostener con estos datos</h2>
  <ol class="conclusions">__CONCLUSIONES__</ol>
</section>

<section id="referencias">
  <div class="eyebrow">08 · Referencias</div>
  <h2>Fuentes estadísticas y conceptuales</h2>
  <ol class="references">
    <li><b>INDEC.</b> Censo Nacional de Población, Hogares y Viviendas 2022. Cuadros utilizados: población por partido, edades simples, índice de envejecimiento y dependencia potencial. <a href="__URL_POBLACION__" target="_blank">Cuadro de población</a> · <a href="__URL_EDADES__" target="_blank">Edades</a> · <a href="__URL_ENVEJECIMIENTO__" target="_blank">Envejecimiento</a> · <a href="__URL_DEPENDENCIA__" target="_blank">Dependencia</a>.</li>
    <li><b>Registro Provincial de las Personas, Provincia de Buenos Aires.</b> Registros mensuales de nacimientos y defunciones, enero de 2020 a septiembre de 2026. <a href="__URL_NAC__" target="_blank">Nacimientos</a> · <a href="__URL_DEF__" target="_blank">Defunciones</a>.</li>
    <li><b>Naciones Unidas.</b> <i>World Population Prospects 2024</i>, utilizado como referencia conceptual para el contexto de transición demográfica. <a href="https://population.un.org/wpp/" target="_blank">Sitio oficial</a>.</li>
    <li><b>UNFPA.</b> <i>Ageing</i>, referencia conceptual sobre envejecimiento poblacional. <a href="https://www.unfpa.org/ageing" target="_blank">Sitio oficial</a>.</li>
  </ol>
  <p class="note">El análisis bonaerense se construye con las fuentes de INDEC y del Registro Provincial. Las referencias internacionales se incluyen como marco conceptual, no para reemplazar los resultados locales.</p>
</section>

<footer>135 partidos · Análisis descriptivo · Corte documental: septiembre de 2026</footer>
</main>

<script>
const datosMunicipios = __DATOS_MUNICIPIOS__;
const datosScatter = __DATOS_SCATTER__;
const datosProvinciales = __DATOS_PROVINCIALES__;
const dependenciaProv = __DEPENDENCIA_PROV__;
const topDependencia = __TOP_DEPENDENCIA__;

const configPlot = {responsive:true, displaylogo:false};
const baseLayout = {
  paper_bgcolor:'#151f30',plot_bgcolor:'#151f30',
  font:{family:'Arial, sans-serif',color:'#f1f5f9',size:13},
  margin:{l:70,r:28,t:55,b:65},
  xaxis:{gridcolor:'#334155',linecolor:'#334155',zerolinecolor:'#475569'},
  yaxis:{gridcolor:'#334155',linecolor:'#334155',zerolinecolor:'#475569'},
  legend:{orientation:'h',y:1.12,x:0}
};

// Evolución provincial
Plotly.newPlot('chart-provincial',[
  {x:datosProvinciales.map(d=>d.anio),y:datosProvinciales.map(d=>d.nacimientos),type:'scatter',mode:'lines+markers',name:'Nacimientos inscriptos',line:{color:'#60a5fa',width:3},marker:{size:8}},
  {x:datosProvinciales.map(d=>d.anio),y:datosProvinciales.map(d=>d.defunciones),type:'scatter',mode:'lines+markers',name:'Defunciones inscriptas',line:{color:'#fb923c',width:3},marker:{size:8}}
],{
  ...baseLayout,
  title:{text:'Evolución anual de nacimientos y defunciones inscriptos',font:{size:18}},
  hovermode:'x unified',
  xaxis:{...baseLayout.xaxis,title:'Año',dtick:1},
  yaxis:{...baseLayout.yaxis,title:'Inscripciones'}
},configPlot);

// Dependencia provincial 2022
Plotly.newPlot('chart-dependencia-prov',[
  {x:['Dependencia joven'],y:[dependenciaProv.joven_2022],type:'bar',name:'0–14',marker:{color:'#60a5fa'},text:[dependenciaProv.joven_2022.toFixed(1)],textposition:'outside'},
  {x:['Dependencia de mayores'],y:[dependenciaProv.mayores_2022],type:'bar',name:'65+',marker:{color:'#2dd4bf'},text:[dependenciaProv.mayores_2022.toFixed(1)],textposition:'outside'}
],{
  ...baseLayout,
  title:{text:'Componentes de la dependencia potencial · PBA 2022',font:{size:16}},
  showlegend:false,
  xaxis:{...baseLayout.xaxis,title:''},
  yaxis:{...baseLayout.yaxis,title:'Personas por cada 100 de 15–64',rangemode:'tozero'}
},configPlot);

// Partidos con mayor dependencia de mayores
Plotly.newPlot('chart-dependencia-top',[{
  x:topDependencia.map(d=>d.dependencia_mayores_2022),
  y:topDependencia.map(d=>d.municipio_nombre),
  type:'bar',orientation:'h',
  marker:{color:'#2dd4bf'},
  text:topDependencia.map(d=>d.dependencia_mayores_2022.toFixed(1)),
  textposition:'outside',
  hovertemplate:'<b>%{y}</b><br>65+ por cada 100 de 15–64: %{x:.1f}<extra></extra>'
}],{
  ...baseLayout,
  title:{text:'10 partidos con mayor dependencia de personas mayores · 2022',font:{size:16}},
  xaxis:{...baseLayout.xaxis,title:'65+ por cada 100 de 15–64'},
  yaxis:{...baseLayout.yaxis,automargin:true},
  margin:{l:145,r:35,t:55,b:55},
  showlegend:false
},configPlot);

// Scatter territorial: índice oficial de envejecimiento
const xs = datosScatter.map(d=>d.indice_envejecimiento_oficial_2022);
const ys = datosScatter.map(d=>d.variacion_nacimientos_pct_2022_2025);
const mediana = arr => {
  const a=[...arr].sort((a,b)=>a-b), m=Math.floor(a.length/2);
  return a.length%2 ? a[m] : (a[m-1]+a[m])/2;
};
const medX=mediana(xs), medY=mediana(ys);

Plotly.newPlot('chart-scatter',[{
  x:xs,
  y:ys,
  text:datosScatter.map(d=>d.municipio_nombre),
  customdata:datosScatter.map(d=>[d.poblacion_2022,d.dependencia_mayores_2022]),
  mode:'markers',
  type:'scatter',
  marker:{
    size:datosScatter.map(d=>d.tamano_burbuja),
    color:datosScatter.map(d=>d.indice_envejecimiento_oficial_2022),
    colorscale:'Viridis',
    showscale:true,
    colorbar:{title:'Índice<br>2022',thickness:14},
    line:{color:'#dbeafe',width:0.7},
    opacity:0.82
  },
  hovertemplate:'<b>%{text}</b><br>Índice de envejecimiento: %{x:.1f}<br>Variación nacimientos: %{y:.1f}%<br>Dependencia de mayores: %{customdata[1]:.1f}<br>Población 2022: %{customdata[0]:,.0f}<extra></extra>'
}],{
  ...baseLayout,
  title:{text:'Nacimientos inscriptos 2022–2025 e índice de envejecimiento 2022',font:{size:18}},
  xaxis:{...baseLayout.xaxis,title:'Índice de envejecimiento 2022 · 65+ por cada 100 menores de 15'},
  yaxis:{...baseLayout.yaxis,title:'Variación de nacimientos inscriptos 2022–2025 (%)',zeroline:true},
  shapes:[
    {type:'line',x0:medX,x1:medX,y0:Math.min(...ys),y1:Math.max(...ys),line:{color:'#64748b',width:1,dash:'dot'}},
    {type:'line',x0:Math.min(...xs),x1:Math.max(...xs),y0:medY,y1:medY,line:{color:'#64748b',width:1,dash:'dot'}}
  ],
  annotations:[
    {x:medX,y:Math.max(...ys),text:'Mediana envejecimiento',showarrow:false,yshift:12,font:{color:'#94a3b8',size:11}},
    {x:Math.max(...xs),y:medY,text:'Mediana variación',showarrow:false,xanchor:'right',yshift:10,font:{color:'#94a3b8',size:11}}
  ]
},configPlot);

// Explorador municipal
const selector=document.getElementById('partido-selector');
datosMunicipios.forEach(d=>{
  const o=document.createElement('option');
  o.value=d.id;o.textContent=d.nombre;selector.appendChild(o);
});

function fmt(v,d=0){
  if(v===null || v===undefined || Number.isNaN(Number(v))) return 'Sin dato';
  return Number(v).toLocaleString('es-AR',{maximumFractionDigits:d,minimumFractionDigits:d});
}

function actualizar(id){
  const mun=datosMunicipios.find(d=>d.id===id);
  if(!mun) return;

  document.getElementById('partido-info-panel').innerHTML=`
    <h3 style="color:#2dd4bf;margin-top:0">${mun.nombre}</h3>
    <p><b>Población censada 2022:</b> ${fmt(mun.poblacion)}</p>
    <p><b>Índice de envejecimiento:</b> ${fmt(mun.indice2010)} en 2010 → <b>${fmt(mun.indice2022)}</b> en 2022.</p>
    <p><b>Dependencia potencial 2022:</b> ${fmt(mun.dep_total,1)} personas de 0–14 y 65+ por cada 100 de 15–64.</p>
    <p><b>Variación de nacimientos inscriptos 2022–2025:</b> ${fmt(mun.cambio,1)} %.</p>
    <p class="note">La dependencia potencial describe grupos de edad; no mide cantidad real de personas ocupadas o inactivas.</p>`;

  Plotly.react('chart-dependencia-partido',[
    {x:[mun.dep_joven],y:['Dependencia potencial'],type:'bar',orientation:'h',name:'0–14',marker:{color:'#60a5fa'},text:[mun.dep_joven.toFixed(1)],textposition:'inside',insidetextanchor:'middle'},
    {x:[mun.dep_mayores],y:['Dependencia potencial'],type:'bar',orientation:'h',name:'65+',marker:{color:'#2dd4bf'},text:[mun.dep_mayores.toFixed(1)],textposition:'inside',insidetextanchor:'middle'}
  ],{
    ...baseLayout,
    barmode:'stack',
    title:{text:'Composición de la dependencia potencial · 2022',font:{size:16}},
    xaxis:{...baseLayout.xaxis,title:'Personas por cada 100 de 15–64',range:[0,Math.max(70,mun.dep_joven+mun.dep_mayores+8)]},
    yaxis:{...baseLayout.yaxis,title:'',showticklabels:false},
    margin:{l:35,r:25,t:55,b:55},
    annotations:[{
      x:mun.dep_joven+mun.dep_mayores,
      y:'Dependencia potencial',
      text:`Total calculado: ${(mun.dep_joven+mun.dep_mayores).toFixed(1)}`,
      showarrow:false,xanchor:'right',yshift:35,font:{color:'#f1f5f9'}
    }]
  },configPlot);

  const serie=mun.series_historicas;
  Plotly.react('chart-nacimientos-partido',[
    {x:serie.map(s=>s.anio),y:serie.map(s=>s.nacimientos),type:'scatter',mode:'lines+markers',name:'Nacimientos inscriptos',line:{color:'#60a5fa',width:3}},
    {x:serie.map(s=>s.anio),y:serie.map(s=>s.defunciones),type:'scatter',mode:'lines+markers',name:'Defunciones inscriptas',line:{color:'#fb923c',width:2.5}}
  ],{
    ...baseLayout,
    title:{text:`Evolución registral · ${mun.nombre}`,font:{size:17}},
    hovermode:'x unified',
    xaxis:{...baseLayout.xaxis,title:'Año',dtick:1},
    yaxis:{...baseLayout.yaxis,title:'Inscripciones'}
  },configPlot);
}

selector.addEventListener('change',e=>actualizar(e.target.value));
selector.value=datosMunicipios.some(d=>d.id==='06427')?'06427':datosMunicipios[0].id;
actualizar(selector.value);
</script>
</body>
</html>
"""

reemplazos = {
    "__PROBLEMATICA__": html.escape(PROBLEMATICA),
    "__OBJETIVO__": html.escape(OBJETIVO),
    "__PREGUNTA__": html.escape(PREGUNTA_ANALISIS),
    "__CAMBIO_NAC__": fmt_pct(cambio_20_25),
    "__NAC_2020__": fmt_int(nac_2020),
    "__NAC_2025__": fmt_int(nac_2025),
    "__PARTIDOS_CAIDA__": str(partidos_caida),
    "__IND_2010__": fmt_num(INDICE_PROV_2010),
    "__IND_2022__": fmt_num(INDICE_PROV_2022),
    "__DEP_2010__": fmt_num(DEPENDENCIA_PROV_2010),
    "__DEP_2022__": fmt_num(DEPENDENCIA_PROV_2022),
    "__DEP_JOVEN__": fmt_pct(dep_joven_prov).replace(" %",""),
    "__DEP_MAYOR__": fmt_pct(dep_mayor_prov).replace(" %",""),
    "__DEP_SUBE__": str(partidos_dep_sube),
    "__DEP_BAJA__": str(partidos_dep_baja),
    "__CONCLUSIONES__": conclusiones_html,
    "__URL_POBLACION__": URL_POBLACION,
    "__URL_EDADES__": URL_EDADES,
    "__URL_ENVEJECIMIENTO__": URL_ENVEJECIMIENTO,
    "__URL_DEPENDENCIA__": URL_DEPENDENCIA,
    "__URL_NAC__": URL_NAC,
    "__URL_DEF__": URL_DEF,
    "__DATOS_MUNICIPIOS__": json_distritos_web,
    "__DATOS_SCATTER__": relacion_json,
    "__DATOS_PROVINCIALES__": provincial_json,
    "__DEPENDENCIA_PROV__": dependencia_prov_json,
    "__TOP_DEPENDENCIA__": top_dependencia_json,
}
html_web_interactivo = html_template
for clave, valor in reemplazos.items():
    html_web_interactivo = html_web_interactivo.replace(clave, valor)

INDEX_HTML.write_text(html_web_interactivo, encoding="utf-8")
print(f"✔ Dashboard generado: {INDEX_HTML}")


## 10. Verificación final


In [ ]:
# Verificación final y archivos de salida
if not INDEX_HTML.exists() or INDEX_HTML.stat().st_size == 0:
    raise RuntimeError("No se generó docs/index.html correctamente.")

contenido = INDEX_HTML.read_text(encoding="utf-8")
controles = {
    "HTML completo": "</html>" in contenido.lower(),
    "Plotly": "Plotly.newPlot" in contenido,
    "Problemática y objetivo": 'id="planteo"' in contenido,
    "Marco teórico": 'id="marco"' in contenido,
    "Análisis de dependencia": 'id="dependencia"' in contenido,
    "Scatter de envejecimiento": "indice_envejecimiento_oficial_2022" in ''.join(relacion_filtrada.columns),
    "Explorador de partidos": "partido-selector" in contenido,
    "Conclusiones": 'id="conclusiones"' in contenido,
    "Referencias": 'id="referencias"' in contenido,
    "135 partidos incorporados": len(filas_json) == 135,
}

for nombre, ok in controles.items():
    print(("✔" if ok else "✘"), nombre)

if not all(controles.values()):
    raise RuntimeError("El HTML no superó todos los controles finales.")

zip_path = shutil.make_archive("docs", "zip", root_dir=DOCS_DIR)

print(f"\n✔ index.html: {INDEX_HTML.stat().st_size / 1024:.1f} KB")
print(f"✔ ZIP: {zip_path}")

display(HTML("<h3>Archivos listos</h3>"))
display(FileLink(str(INDEX_HTML)))
display(FileLink(zip_path))
